## 📘 Basic Agent
**Goal**: Create an agent using the **Foundry Project SDK** (`azure-ai-projects`) so it is registered server-side with full **tracing in the Foundry portal**, then connect to it from the **Microsoft Agent Framework** via `FoundryAgent`.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")  # for AIProjectClient
MODEL                 = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

AGENT_NAME = "ContosoConcierge-af"

print(f"📡 Project endpoint : {PROJECT_ENDPOINT}")
print(f"🧠 Model:                   {MODEL}")

📡 Project endpoint : https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model:                   gpt-4o


---
### Part A — Create Agent with Foundry Project SDK 🏗️


In [2]:
# ── A1. Create the agent in Foundry ──────────────────────────────────────────
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

INSTRUCTIONS = """\
You are **Contoso Concierge**, a friendly and knowledgeable assistant for
Contoso Corporation. You help employees with general questions about company
policies, product information, and day-to-day tasks.

Guidelines:
- Be concise but thorough.
- When you don't know something, say so honestly.
- Use a professional yet warm tone.
"""

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

agent_version = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=INSTRUCTIONS,
    ),
)

print(f"✅ Agent created in Foundry!")
print(f"   Name:    {agent_version.name}")
print(f"   Version: {agent_version.version}")
print(f"   ID:      {agent_version.id}")
print(f"\n🔍 Traces for this agent will appear in the Foundry portal.")

✅ Agent created in Foundry!
   Name:    ContosoConcierge-af
   Version: 1
   ID:      ContosoConcierge-af:1

🔍 Traces for this agent will appear in the Foundry portal.


---
### Part B — Use with Microsoft Agent Framework 🤖

In [3]:
# ── B1. Connect to the Foundry agent via Agent Framework ─────────────────────
from agent_framework import Agent
from agent_framework.foundry import FoundryAgent

# af_agent = FoundryAgentChatClient(
#                 project_endpoint=PROJECT_ENDPOINT,
#                 agent_name=AGENT_NAME,
#                 agent_version=str(agent_version.version),
#                 credential=get_credential(),
#             ).get_agent()

af_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=agent_version.name,
    agent_version=str(agent_version.version),
    credential=get_credential(),
)

print(f"🤖 Agent Framework connected to Foundry agent: {agent_version.name} v{agent_version.version}")
print(f"   Type: {type(af_agent).__name__}")

🤖 Agent Framework connected to Foundry agent: ContosoConcierge-af v1
   Type: FoundryAgent


In [4]:
# ── B2. Single-turn call via Agent Framework ─────────────────────────────────
result = await af_agent.run("What departments does Contoso have?")
print(f"🗨️  Agent (via Agent Framework): {result}")

🗨️  Agent (via Agent Framework): Contoso Corporation has several key departments, including:

1. **Human Resources (HR)**: Manages employee relations, benefits, and recruitment.
2. **Finance**: Oversees budgeting, accounting, and financial planning.
3. **Marketing**: Focuses on brand management, advertising, and market research.
4. **Sales**: Handles customer relationships and drives revenue.
5. **Product Development**: Develops new products and improvements.
6. **Information Technology (IT)**: Manages tech infrastructure and support.
7. **Customer Service**: Addresses customer inquiries and support.

If you need more specific details, feel free to ask!


In [6]:
# ── B3. Multi-turn conversation with AgentSession ────────────────────────────
session = af_agent.create_session()
print(f"💬 Session created: {session.session_id}")

# First turn
result1 = await af_agent.run(
    "My name is Dibyendu and I work in IT department. I like chocolates.",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: 35d99c4b-8201-425f-82f9-19e809ca8e6b

🗨️  Agent: Hi Dibyendu! It's great to meet a fellow chocolate enthusiast. If you need any assistance with IT-related matters or anything else, feel free to let me know.

🗨️  Agent: You mentioned that you like chocolates! If there's anything else you enjoy or need help with, just let me know.


In [7]:
# ── B4. Streaming response ───────────────────────────────────────────────────
print("🗨️  Agent (streaming): ", end="", flush=True)
async for chunk in af_agent.run("Tell me how to make a soup.", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

🗨️  Agent (streaming): Certainly! Here's a basic recipe for a simple vegetable soup:

**Ingredients:**
- 2 tablespoons olive oil
- 1 onion, chopped
- 2 carrots, sliced
- 2 celery stalks, sliced
- 3 cloves garlic, minced
- 1 zucchini, chopped
- 1 bell pepper, chopped
- 4 cups vegetable broth
- 1 can (14 oz) diced tomatoes
- 1 teaspoon Italian seasoning
- Salt and pepper to taste
- Optional: 1 cup chopped spinach or kale

**Instructions:**
1. **Prepare Ingredients:** Gather and chop all your vegetables and garlic.

2. **Sauté Aromatics:** In a large pot, heat olive oil over medium heat. Add the onion, carrots, and celery. Cook for about 5 minutes until softened. Add garlic and sauté for another minute.

3. **Add Vegetables:** Stir in the zucchini and bell pepper. Cook for another 5 minutes.

4. **Add Liquids and Seasoning:** Pour in the vegetable broth and diced tomatoes (with their juice). Add Italian seasoning, salt, and pepper.

5. **Simmer:** Bring the mixture to a boil, then reduce 

---
### Part C — Cleanup 🧹

In [8]:
# ── C1. Delete the agent version from Foundry ────────────────────────────────
# Uncomment the next line to delete the agent when you're done:
project.agents.delete(agent_name=AGENT_NAME)
print("🧹 Agent deleted from Foundry")

#print("💡 Tip: Keep the agent to reuse in other notebooks or the Foundry Playground.")
print(f"   Agent name:    {AGENT_NAME}")
print(f"   Agent version: {agent_version.version}")

🧹 Agent deleted from Foundry
   Agent name:    ContosoConcierge-af
   Agent version: 1


---
### ✅ What you demonstrated

| Capability | SDK / Feature |
|---|---|
| Register agent in Foundry | `AIProjectClient` → `agents.create_version()` |
| Server-side traces | All calls visible in Foundry portal → Tracing |
| Agent Framework bridge | `FoundryAgent(agent_name=..., agent_version=...)` |
| Multi-turn memory | `AgentSession` retains context across turns |
| Streaming | `async for chunk in agent.run(..., stream=True)` |

**Next →** [02_tool_augmented_agent.ipynb](./02_tool_augmented_agent.ipynb) — Code Interpreter, File Search, Web Search & custom Function Tools.